In [25]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

d = fetch_ucirepo(id=967)  # download do dataset
df = pd.concat([d.data.features, d.data.targets], axis=1)   # tabela com caracteristicas do url e target

df["phishing"] = 1 - df["label"]        # no dataset original 1 = legítimo, 0 = phishing; inversão para ficar intuitivo

df.to_csv("../data/phiusiil.csv", index=False)
print(df.shape)                         # numero linhas e colunas
df["phishing"].value_counts()           # quais são phishing (1) e quais não (0)

(235795, 56)


phishing
0    134850
1    100945
Name: count, dtype: int64

In [26]:
df[["URL", "URLLength", "NoOfSubDomain", "IsHTTPS", "phishing"]].sample(10)

,URL,URLLength,NoOfSubDomain,IsHTTPS,phishing
230315,https://www.moog.com,19,1,1,0
58877,http://www.payymdcotor.com,25,1,0,1
14620,https://www.perfectsettings.com,30,1,1,0
28519,https://www.cityoftracy.org,26,1,1,0
203146,https://www.saao.ac.za,21,2,1,0
203463,https://www.torakuru.jp,22,1,1,0
102120,http://www.selfdefansakademi.com,32,1,0,1
105700,https://www.tour-dubai.com,25,1,1,0
211228,https://www.elcode.ru,20,1,1,0
128814,http://olx.2383659.xyz/xc8mtqxe/technical-works,47,1,0,1


In [27]:
from urllib.parse import urlparse
import math
from collections import Counter

PALAVRAS = ["verify", "secure", "account", "update", "bank"]

MARCAS = {
    "facebook": "facebook.com",
    "instagram": "instagram.com",
    "linkedin": "linkedin.com",
    "dhl": "dhl.com",
    "docusign": "docusign.com",
    "dropbox": "dropbox.com",
    "ctt": "ctt.pt",
    "santander": "santander.pt",
    "millenniumbcp": "millenniumbcp.pt",
}

def marca_falsa(url, host):
    host = host.split("@")[-1].split(":")[0].lower()
    for marca, dominio in MARCAS.items():
        if marca in url.lower():
            if host != dominio and not host.endswith("." + dominio):
                return 1
    return 0


def entropia(s):
    # mede o "quão aleatório" é o texto (domínios gerados por máquina têm valores altos)
    if not s:
        return 0
    c = Counter(s)
    return -sum(n / len(s) * math.log2(n / len(s)) for n in c.values())  # entropia de Shannon



def extrair_features(url):
    url = url.replace("://www.", "://", 1)
    p = urlparse(url)
    host = p.netloc  # ex.: "paypa1-login.xyz"
    return {
        "comprimento": len(url),
        "n_pontos": host.count("."),
        "n_hifenes": host.count("-"),
        "n_digitos": sum(c.isdigit() for c in url),
        "tem_https": int(p.scheme == "https"),
        "tem_arroba": int("@" in url),
        "tem_ip": int(host.replace(".", "").isdigit()),
        "tem_login": int("login" in url.lower()),
        "entropia": entropia(host),
        "n_palavras_suspeitas": sum(w in url.lower() for w in PALAVRAS),
        "tem porta": int(":" in host.split("@")[-1]),
        "marca_falsa": marca_falsa(url, host)

    }

extrair_features("https://paypa1-login-secure.xyz/verify")


{'comprimento': 38,
 'n_pontos': 1,
 'n_hifenes': 2,
 'n_digitos': 1,
 'tem_https': 1,
 'tem_arroba': 0,
 'tem_ip': 0,
 'tem_login': 1,
 'entropia': 4.088779347361361,
 'n_palavras_suspeitas': 2,
 'tem porta': 0,
 'marca_falsa': 0}

In [28]:
X = pd.DataFrame([extrair_features(u) for u in df["URL"]])
y = df["phishing"]
X.head()

,comprimento,n_pontos,n_hifenes,n_digitos,tem_https,tem_arroba,tem_ip,tem_login,entropia,n_palavras_suspeitas,tem porta,marca_falsa
0,28,1,0,0,1,0,0,0,3.546439,1,0,0
1,20,1,1,0,1,0,0,0,3.251629,0,0,0
2,26,2,0,0,1,0,0,0,3.572431,0,0,0
3,23,1,0,0,1,0,0,0,3.506891,0,0,0
4,30,1,0,0,1,0,0,0,3.413088,0,0,0


In [29]:
from sklearn.model_selection import train_test_split                        # divide os dados em treino e teste
from xgboost import XGBClassifier                                           # modelo classificador
from sklearn.metrics import classification_report, confusion_matrix         # medidor qualidade


# etapa 1 - Dividir os Dados
# X = as pistas (features), y = a resposta certa (0 legítimo, 1 phishing)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,               # o que vamos dividir
    test_size=0.2,      # 20% vão para teste, 80% para treino
    stratify=y, 
    random_state=42
)

# X_train/y_train: o que o modelo vai estudar
# X_test/y_test:   o que ele nunca viu (teste surpresa)


# 2. treinar
model = XGBClassifier(n_estimators=200, max_depth=6, random_state=42)   # nº de árvores de decisão, profundidade máxima cada, resultados repetiveis
model.fit(X_train, y_train)              # o modelo estuda: aprende que pistas levam a que resposta

# 3. testar
pred = model.predict(X_test)
print(classification_report(y_test, pred, target_names=["legítimo", "phishing"]))     # compara as respostas dele (pred) com as certas (y_test)
print(confusion_matrix(y_test, pred))

              precision    recall  f1-score   support

    legítimo       0.98      0.99      0.98     26970
    phishing       0.98      0.97      0.98     20189

    accuracy                           0.98     47159
   macro avg       0.98      0.98      0.98     47159
weighted avg       0.98      0.98      0.98     47159

[[26668   302]
 [  662 19527]]


In [30]:
pd.Series(
    model.feature_importances_,   # quanto cada pista pesou nas decisões do modelo
    index=X.columns               # associa cada número ao nome da pista
).sort_values(ascending=False)    # ordena da mais importante para a menos

tem_https               0.933543
comprimento             0.021879
n_digitos               0.019468
n_pontos                0.006879
tem_login               0.005958
n_palavras_suspeitas    0.005130
marca_falsa             0.002888
n_hifenes               0.002020
entropia                0.001837
tem_arroba              0.000398
tem_ip                  0.000000
tem porta               0.000000
dtype: float32

In [41]:
pd.crosstab(X["tem_https"], y, normalize="index")

phishing,0,1
tem_https,,
0,0.000000,1.000000
1,0.732697,0.267303


In [32]:
X2 = X.drop(columns=["tem_https"])
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.2, stratify=y, random_state=42)
m2 = XGBClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X2_train, y2_train)
print(classification_report(y2_test, m2.predict(X2_test), target_names=["legítimo", "phishing"]))

              precision    recall  f1-score   support

    legítimo       0.94      0.98      0.96     26970
    phishing       0.97      0.92      0.95     20189

    accuracy                           0.95     47159
   macro avg       0.96      0.95      0.95     47159
weighted avg       0.95      0.95      0.95     47159



In [33]:
import os
os.listdir("../data")

['phishbench_external.csv',
 'PhishBench-External Balanced Phishing and Legitima',
 'phiusiil.csv']

In [34]:
ext = pd.read_csv("../data/phishbench_external.csv")
print(ext.shape)
ext.columns.tolist()

(1000, 29)


['URL',
 'Using the IP Address',
 'Long URL',
 'URL Shortening Services',
 'URL having @ Symbol',
 'Redirecting using //',
 'Prefix/Suffix with - in Domain',
 'Sub Domain and Multi Sub Domains',
 'HTTPS',
 'Domain Registration Length',
 'Favicon',
 'HTTPS Token in Domain',
 'Request URL',
 'URL of Anchor',
 'Links in Meta/Script/Link Tags',
 'Server Form Handler (SFH)',
 'Submitting to Email',
 'Abnormal URL',
 'Website Forwarding',
 'Status Bar Customization',
 'Using Pop-up Window',
 'Age of Domain',
 'DNS Record',
 'Website Traffic',
 'PageRank',
 'Google Index',
 'Number of Links Pointing to Page',
 'Statistical-Reports Based Feature',
 'Results']

In [35]:
print(ext["Results"].value_counts())
ext["URL"].head(10)

Results
1    500
0    500
Name: count, dtype: int64


0       https://storage739qh83jqutg3fds982.weebly.com/
1           https://icewarpwebclientbright.weebly.com/
2                     https://homeservi5r4.weebly.com/
3    https://xn--generalitatdecatalunyainformacidem...
4          https://cranmerkathie.wixsite.com/my-site-1
5                     https://notacustomer.weebly.com/
6                        https://kataitaly.weebly.com/
7                         https://00192223.weebly.com/
8    https://blueshtt196cpsess694100038883rdparttyr...
9                         https://mmurjtyu.weebly.com/
Name: URL, dtype: str

In [36]:
ext["URL"].str.startswith(("http://", "https://")).value_counts()

URL
True    1000
Name: count, dtype: int64

In [37]:
X_ext = pd.DataFrame([extrair_features(u) for u in ext["URL"]])
y_ext = ext["Results"]
pred_ext = model.predict(X_ext)
print(classification_report(y_ext, pred_ext, target_names=["legítimo", "phishing"]))
print(confusion_matrix(y_ext, pred_ext))

              precision    recall  f1-score   support

    legítimo       0.93      1.00      0.96       500
    phishing       1.00      0.92      0.96       500

    accuracy                           0.96      1000
   macro avg       0.96      0.96      0.96      1000
weighted avg       0.96      0.96      0.96      1000

[[499   1]
 [ 39 461]]


In [38]:
print(df[df["phishing"] == 0]["URL"].sample(10, random_state=1).tolist())
print(ext[ext["Results"] == 0]["URL"].sample(10, random_state=1).tolist())

['https://www.levelup.com', 'https://www.notjusttoyz.com', 'https://www.discover-suriname.com', 'https://www.ecolabelindex.com', 'https://www.cinematheque.fr', 'https://www.necclassicmotorshow.com', 'https://www.ringling.org', 'https://www.hlcommission.org', 'https://www.divx-digest.com', 'https://www.smartgecko.co.za']
['https://consultant.ru', 'https://dell.com', 'https://dns.google', 'https://macromedia.com', 'https://buzzfeed.com', 'https://conviva.com', 'https://bilibili.com', 'https://outlook.com', 'https://wsj.com', 'https://soundcloud.com']


In [39]:
print(pd.crosstab(df["URL"].str.contains("://www."), df["phishing"], normalize="columns"))
print(pd.crosstab(ext["URL"].str.contains("://www."), ext["Results"], normalize="columns"))

phishing    0         1
URL                    
False     0.0  0.583001
True      1.0  0.416999
Results      0     1
URL                 
False    0.998  0.98
True     0.002  0.02


In [40]:
print(pd.crosstab(X["tem_https"], y, normalize="columns"))
print(pd.crosstab(X_ext["tem_https"], y_ext, normalize="columns"))

phishing     0         1
tem_https               
0          0.0  0.512646
1          1.0  0.487354
Results      0      1
tem_https            
0          0.0  0.154
1          1.0  0.846


In [42]:
X2_ext = X_ext.drop(columns=["tem_https"])
pred2_ext = m2.predict(X2_ext)
print(classification_report(y_ext, pred2_ext, target_names=["legítimo", "phishing"]))
print(confusion_matrix(y_ext, pred2_ext))

              precision    recall  f1-score   support

    legítimo       0.90      0.98      0.94       500
    phishing       0.98      0.89      0.93       500

    accuracy                           0.94      1000
   macro avg       0.94      0.94      0.93      1000
weighted avg       0.94      0.94      0.93      1000

[[491   9]
 [ 56 444]]


In [43]:
import os
os.makedirs("../models", exist_ok=True)
model.save_model("../models/xgb_phishing.json")

from xgboost import XGBClassifier
m_carregado = XGBClassifier()
m_carregado.load_model("../models/xgb_phishing.json")
print((m_carregado.predict(X_ext) == pred_ext).all())

True
